# ASX Announcement Tone Engine — Step 1

**Question:** does the tone of an ASX results announcement predict the share price reaction?

**How to use**
1. Works in Google Colab or Jupyter. In Colab: `Runtime → Change runtime type → T4 GPU` (optional, makes FinBERT faster).
2. Download results announcements (PDFs) from the ASX website and rename them `TICKER_YYYY-MM-DD.pdf`, e.g. `BHP_2026-08-19.pdf` (date = the day it was released).
3. Run each cell top to bottom (`Shift + Enter`). Cell 3 asks you to upload the PDFs.

**What it does:** extracts the text → scores every sentence with FinBERT → gives each announcement a tone score → pulls the share price move vs the ASX 200 → charts tone against the move.

## 1. Install libraries

In [ ]:
%pip install -q transformers torch yfinance pdfplumber requests

## 2. Imports and settings

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import pdfplumber
import yfinance as yf
import matplotlib.pyplot as plt
import torch
from transformers import pipeline

FOLDER = "announcements"   # where the PDFs live
INDEX = "^AXJO"            # ASX 200, used as the market benchmark
WINDOW = (-1, 1)           # trading days around the announcement: day before to day after
os.makedirs(FOLDER, exist_ok=True)

## 3A. Fetch announcements automatically
Type the tickers you want. For each one, this finds its most recent results announcements (price-sensitive, from this year and last) and saves them as `TICKER_YYYY-MM-DD.pdf`.

It uses the ASX's public announcement pages, which aren't an official API, so if the ASX changes its site this cell may break. If it does, use 3B instead.

In [ ]:
import requests, time
from html import unescape

TICKERS = ["BHP", "CBA", "CSL", "NAB", "WBC", "ANZ", "WES", "MQG", "GMG", "RIO",
           "WOW", "TLS", "FMG", "WDS", "TCL", "ALL", "REA", "COL", "QBE", "STO"]
PER_TICKER = 2        # how many recent results announcements per company
YEARS = [pd.Timestamp.today().year, pd.Timestamp.today().year - 1]

S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (student research project)"})
BASE = "https://www.asx.com.au"

EXCLUDE = ["meeting", "agm", "email", "letter", "shareholder", "items impacting", "presentation",
           "webcast", "teleconference", "briefing", "investor", "dividend", "distribution",
           " nz", "new zealand", "asb ", "bnz", "date of", "timing", "notice", "pre-result",
           "preliminary final report", "trading update", "transcript", "quarter", "production"]
PERIOD = re.compile(r"\b(fy|hy|1h|2h)\s?\d*\b|full[- ]year|half[- ]year|interim|annual|financial year")

def result_rank(headline):
    # Higher = better match for a company's own results release. 0 = skip.
    h = " " + headline.lower() + " "
    if any(w in h for w in EXCLUDE):
        return 0
    period = bool(PERIOD.search(h))
    if "result" in h and period and any(w in h for w in ["announcement", "release", "media", "asx"]):
        return 4
    if "appendix 4e" in h or "appendix 4d" in h:
        return 3
    if "result" in h and period:
        return 3
    if "profit" in h and period:
        return 2
    return 0

def parse_listing(html):
    rows = []
    for tr in re.split(r"<tr[\s>]", html)[1:]:
        date = re.search(r"(\d{2}/\d{2}/\d{4})", tr)
        link = re.search(r'href="([^"]*displayAnnouncement\.do[^"]*idsId=(\d+)[^"]*)"[^>]*>\s*(.*?)\s*<', tr, re.S)
        if not (date and link):
            continue
        rows.append({
            "date": pd.to_datetime(date.group(1), dayfirst=True),
            "url": BASE + unescape(link.group(1)) if link.group(1).startswith("/") else unescape(link.group(1)),
            "headline": re.sub(r"\s+", " ", unescape(link.group(3))).strip(),
            "price_sensitive": "pricesens" in tr.lower() and "<img" in tr.lower(),
        })
    return rows

def list_announcements(ticker):
    rows = []
    for y in YEARS:
        url = (f"{BASE}/asx/v2/statistics/announcements.do?by=asxCode&asxCode={ticker}"
               f"&timeframe=Y&year={y}")
        r = S.get(url, timeout=30)
        r.raise_for_status()
        rows += parse_listing(r.text)
        time.sleep(1)
    return rows

def pick_results(rows, n=PER_TICKER):
    # One document per results season (best headline within ~6 weeks), most recent first.
    cands = [dict(r, rank=result_rank(r["headline"])) for r in rows]
    cands = sorted([c for c in cands if c["rank"] >= 2], key=lambda c: c["date"], reverse=True)
    picks = []
    for c in cands:
        if picks and (picks[-1]["date"] - c["date"]).days <= 45:
            if c["rank"] > picks[-1]["rank"]:
                picks[-1] = c
        else:
            picks.append(c)
    return picks[:n]

def download_pdf(url, path):
    r = S.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        # ASX shows a terms page first; the real PDF link is inside it
        m = (re.search(r'name="pdfURL"\s+value="([^"]+)"', r.text)
             or re.search(r'(https?://announcements\.asx\.com\.au/asxpdf/[^"\'\s]+\.pdf)', r.text))
        if not m:
            raise RuntimeError("couldn't find the PDF link on the ASX page")
        r = S.get(unescape(m.group(1)), timeout=60)
        r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise RuntimeError("download wasn't a PDF")
    with open(path, "wb") as f:
        f.write(r.content)

# clear out any earlier downloads so wrong files don't hang around
for old in glob.glob(os.path.join(FOLDER, "*.pdf")):
    os.remove(old)

log = []
for t in TICKERS:
    try:
        picks = pick_results(list_announcements(t))
        if not picks:
            print(f"{t}: no results announcements found")
        for p in picks:
            fname = f"{t}_{p['date'].date()}.pdf"
            path = os.path.join(FOLDER, fname)
            if not os.path.exists(path):
                download_pdf(p["url"], path)
                time.sleep(1)
            print(f"{t}: saved {fname}  ({p['headline']})")
            log.append({"ticker": t, "date": p["date"].date(), "headline": p["headline"], "file": fname})
    except Exception as e:
        print(f"{t}: FAILED - {e}")

pd.DataFrame(log)

**Check the list above.** Each headline should be a results release. If one isn't (e.g. a dividend notice), delete that PDF from the `announcements` folder (folder icon on the left) before carrying on.

## 3B. Or upload PDFs yourself
Only run this if 3A didn't work, or to add extra files.

In [ ]:
try:
    from google.colab import files   # Colab: pop-up upload button
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(FOLDER, name), "wb") as f:
            f.write(data)
except ImportError:                    # Jupyter: just copy the PDFs into the folder
    print(f"Copy your PDFs into: {os.path.abspath(FOLDER)}")
print(sorted(os.listdir(FOLDER)))

## 4. Extract and clean the text
Pulls the text out of each PDF, splits it into sentences, and drops table rows and fragments (FinBERT only reads proper sentences well).

In [ ]:
def parse_filename(path):
    m = re.match(r"([A-Za-z0-9]+)_(\d{4}-\d{2}-\d{2})", os.path.basename(path))
    if not m:
        raise ValueError(f"Rename {path} to TICKER_YYYY-MM-DD.pdf")
    return m.group(1).upper(), pd.Timestamp(m.group(2))

def pdf_text(path):
    with pdfplumber.open(path) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

def to_sentences(text):
    text = re.sub(r"-\n", "", text)          # rejoin hyphenated line breaks
    text = re.sub(r"\s*\n\s*", " ", text)    # join lines
    text = re.sub(r"\s{2,}", " ", text)
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    keep = []
    for s in parts:
        s = s.strip()
        words = s.split()
        if len(words) < 6 or len(words) > 80:
            continue
        letters = sum(c.isalpha() for c in s)
        if letters / max(len(s), 1) < 0.6:   # mostly numbers = table row
            continue
        keep.append(s)
    return keep

docs = []
for path in sorted(glob.glob(os.path.join(FOLDER, "*.pdf"))):
    ticker, date = parse_filename(path)
    sents = to_sentences(pdf_text(path))
    docs.append({"ticker": ticker, "date": date, "file": os.path.basename(path), "sentences": sents})
    print(f"{ticker} {date.date()}: {len(sents)} sentences")

## 5. Load FinBERT
A free, open-source model trained on financial text. It labels each sentence **positive**, **negative** or **neutral**. No API key needed.

In [ ]:
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    truncation=True,
    device=0 if torch.cuda.is_available() else -1,
)
finbert("Revenue increased 12% on the prior year, driven by strong demand.")

## 6. Score each announcement
For each sentence: **net tone = P(positive) − P(negative)**, so it runs from −1 to +1.

For each announcement:
- `tone_net`: average net tone across all sentences
- `pct_pos` / `pct_neg`: share of sentences labelled positive / negative
- `tone_1_10`: `tone_net` rescaled to 1–10 for the eventual website

In [ ]:
def score_sentences(sents, batch_size=32):
    out = finbert(sents, batch_size=batch_size)
    rows = []
    for s, res in zip(sents, out):
        p = {r["label"].lower(): r["score"] for r in res}
        rows.append({"sentence": s, "pos": p.get("positive", 0), "neg": p.get("negative", 0),
                     "neu": p.get("neutral", 0)})
    df = pd.DataFrame(rows)
    df["net"] = df["pos"] - df["neg"]
    df["label"] = df[["pos", "neg", "neu"]].idxmax(axis=1)
    return df

def summarise(df):
    tone = df["net"].mean()
    return {
        "n_sentences": len(df),
        "tone_net": tone,
        "pct_pos": (df["label"] == "pos").mean(),
        "pct_neg": (df["label"] == "neg").mean(),
        "tone_1_10": float(np.clip(5.5 + 4.5 * tone, 1, 10)),
    }

sentence_scores = {}
rows = []
for d in docs:
    if not d["sentences"]:
        print("No usable text in", d["file"], "(scanned PDF?) - skipped")
        continue
    df = score_sentences(d["sentences"])
    sentence_scores[d["file"]] = df
    rows.append({"ticker": d["ticker"], "date": d["date"], **summarise(df)})

tone = pd.DataFrame(rows)
tone

### Sanity check
Look at the most positive and negative sentences for one announcement. If these look wrong, the scores can't be trusted.

In [ ]:
first = list(sentence_scores)[0]
df = sentence_scores[first].sort_values("net")
print(first, "\n--- most negative ---")
for s in df.head(3)["sentence"]: print("-", s)
print("--- most positive ---")
for s in df.tail(3)["sentence"]: print("-", s)

## 7. Share price reaction
**Abnormal return** = stock return − ASX 200 return on the same day. This strips out whatever the whole market did.

**CAR** (cumulative abnormal return) adds these up from the day before to the day after the announcement. ASX results usually drop before the open, so day 0 is the release date itself.

In [ ]:
def close_prices(symbol, start, end):
    px = yf.download(symbol, start=start, end=end, auto_adjust=True, progress=False)["Close"]
    if isinstance(px, pd.DataFrame):
        px = px.iloc[:, 0]
    return px.dropna()

def car(ticker, date, window=WINDOW):
    start, end = date - pd.Timedelta(days=20), date + pd.Timedelta(days=20)
    stock = close_prices(f"{ticker}.AX", start, end).pct_change()
    market = close_prices(INDEX, start, end).pct_change()
    ar = (stock - market).dropna()
    if ar.empty:
        return np.nan
    day0 = ar.index.searchsorted(date)   # first trading day on/after release
    lo, hi = day0 + window[0], day0 + window[1]
    if lo < 0 or hi >= len(ar):
        return np.nan
    return ar.iloc[lo:hi + 1].sum()

tone["car"] = [car(t, d) for t, d in zip(tone["ticker"], tone["date"])]
tone["car_pct"] = tone["car"] * 100
tone.round(dict.fromkeys(tone.select_dtypes("number").columns, 3))

## 8. Does tone predict the move?

In [ ]:
res = tone.dropna(subset=["car"])
n = len(res)
r = res["tone_net"].corr(res["car"]) if n > 2 else np.nan

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(res["tone_net"], res["car_pct"], s=60, alpha=0.8)
for _, row in res.iterrows():
    ax.annotate(row["ticker"], (row["tone_net"], row["car_pct"]), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
if n > 2:
    slope, intercept = np.polyfit(res["tone_net"], res["car_pct"], 1)
    xs = np.linspace(res["tone_net"].min(), res["tone_net"].max(), 50)
    ax.plot(xs, slope * xs + intercept, linestyle="--")
ax.axhline(0, linewidth=0.8, color="grey")
ax.set_xlabel("Announcement tone (FinBERT net, −1 to +1)")
ax.set_ylabel(f"Abnormal return, days {WINDOW[0]} to +{WINDOW[1]} (%)")
ax.set_title(f"Tone vs share price reaction  (n = {n}, correlation = {r:.2f})")
plt.tight_layout()
plt.show()

**Reading it:** a positive correlation means more upbeat announcements tended to see better reactions. With ~20 announcements this is noisy, so treat it as a first look, not proof. Results season brings a lot more data.

## 9. Save results

In [ ]:
tone.to_csv("tone_results.csv", index=False)
print("Saved", os.path.abspath("tone_results.csv"))
try:
    from google.colab import files
    files.download("tone_results.csv")
except ImportError:
    pass

## Next steps
- **More data:** add more companies and past results seasons (each company has two a year).
- **Spin score (Step 2):** pull headline numbers (profit change, guidance) and flag when tone is much rosier than the numbers.
- **Website (Step 3):** wrap the engine in a Streamlit page where you paste an announcement and get the scores.